In [1]:
%pip install python-dotenv openai datasets math_verify tqdm torch aiolimiter

  Using cached datasets-5.0.1-py3-none-any.whl.metadata (23 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
  Using cached torch-2.13.0-cp314-cp314-macosx_14_0_arm64.whl.metadata (39 kB)
  Using cached numpy-2.5.2-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached pyarrow-25.0.1-cp314-cp314-macosx_12_0_arm64.whl.metadata (3.0 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached pandas-3.0.5-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached xxhash-4.0.1-cp314-cp314-macosx_11_0_arm64.whl.metadata (17 kB)
  Using cached multiprocess-0.70.19-py314-none-any.whl.metadata (7.2 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached aiohttp-3.14.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (8.3 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached setuptools-84.0.0-py3-none-any.whl.metadat

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import logging

logging.basicConfig(level=logging.INFO)

In [4]:
import os
from openai import AsyncOpenAI
from aiolimiter import AsyncLimiter
from asyncio import Semaphore
from math_verify import parse
from dataclasses import dataclass

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
MODEL_ID = "openai/gpt-oss-20b"

client = AsyncOpenAI(
	base_url="https://integrate.api.nvidia.com/v1",
	api_key=NVIDIA_API_KEY,
	max_retries=0,
)

@dataclass
class MyCompletionChoice:
	reasoning_content: str = ""
	content: str = ""

limiter = AsyncLimiter(20)
semaphore = Semaphore(80)
async def create_completion(*args, **kwargs):
	kwargs["stream"] = True
	while True:
		try:
			async with limiter:
				async with semaphore:
					choices: list[MyCompletionChoice] = []
					async for chunk in await client.chat.completions.create(*args, **kwargs):
						for choice in chunk.choices:
							while len(choices) <= choice.index:
								choices.append(MyCompletionChoice())

							if delta := getattr(choice.delta, "reasoning_content", None):
								choices[choice.index].reasoning_content += delta

							if delta := getattr(choice.delta, "content", None):
								choices[choice.index].content += delta
					return choices
		except:
			pass

prompt = "What is 13 times 17? Box your answer."
gold = "221"

choices = await create_completion(
	model=MODEL_ID,
	messages=[{"role": "user", "content": prompt}],
	max_tokens=2**10,
    n=8,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**10
    },
)

print("*"*20, "Prompt", "*"*20)
print(prompt)

parsed_gold = parse(gold) or [None]
for i, choice in enumerate(choices):
  parsed_answer = parse(choice.content) or [None]
  correct = parsed_gold[0] == parsed_answer[0]

  print("*"*20, f"Choice {i+1}: {parsed_answer[0]} ({'correct' if correct else 'incorrect'})", "*"*20)
  if correct:
    print(f"<think>{choice.reasoning_content}</think>{choice.content}")

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"


******************** Prompt ********************
What is 13 times 17? Box your answer.
******************** Choice 1: 221 (correct) ********************
<think>The user asks: "What is 13 times 17? Box your answer." They likely want the multiplication result: 13 * 17 = 221. "Box your answer" means put it in a box ASCII or maybe format. We can write:

```
221
```

or box around: perhaps 

```
┌────┐
│ 221│
└────┘
```

But simply with braces: [221]. Could do:

<span style="border:1px solid #000;">221</span> but we can't use html. So best is ASCII box:

```
+-----+
| 221 |
+-----+
```

Alternatively:

```
┌─────┐
│ 221 │
└─────┘
```

We can respond with that.</think>The answer is:

```
┌─────┐
│ 221 │
└─────┘
```
******************** Choice 2: 221 (correct) ********************
<think>We need to respond: 13 x 17 = 221. There's instruction to box the answer. Usually they want a box drawn around the number? So something like:

$\boxed{221}$

That should satisfy.</think>\[
\boxed{221}
\]
****

In [5]:
from datasets import load_dataset

ds = load_dataset("open-r1/OpenR1-Math-220k", "default", split="train")
ds

INFO:httpx:HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/open-r1/OpenR1-Math-220k/e4e141ec9dea9f8326f4d347be56105859b2bd68/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/e4e141ec9dea9f8326f4d347be56105859b2bd68/OpenR1-Math-220k.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/open-r1/OpenR1-Math-220k/open-r1/OpenR1-Math-220k.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/open-r1/OpenR1-Math-220k/revision/e4e141ec9dea9f8326f4d347be56105859b2bd68 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-M

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/open-r1/OpenR1-Math-220k/resolve/e4e141ec9dea9f8326f4d347be56105859b2bd68/dataset_infos.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/open-r1/OpenR1-Math-220k/tree/e4e141ec9dea9f8326f4d347be56105859b2bd68/data?recursive=true&expand=false "HTTP/1.1 200 OK"


Dataset({
    features: ['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count', 'messages'],
    num_rows: 93733
})

In [6]:
from tqdm.contrib.logging import logging_redirect_tqdm
from tqdm.asyncio import tqdm_asyncio
from math_verify import verify
from datasets import Dataset

async def generate_dataset(prompts, golds, **kwargs):
	dataset_dict = {
		"prompt": [],
		"outputs": [],
		"results": [],
	}
	futures = []
	for prompt in prompts:
		futures.append(create_completion(
			**kwargs,
			messages=[{"role": "user", "content": prompt}],
		))
	with logging_redirect_tqdm():
		completions = await tqdm_asyncio.gather(*futures, desc="Creating completions")
	for prompt, choices, gold in zip(prompts, completions, golds):
		gold = parse(gold)

		outputs = []
		results = []
		for choice in choices:
			answer = parse(choice.content)
			result = verify(gold, answer)

			outputs.append(f"<think>{choice.reasoning_content}</think>{choice.content}")
			results.append(result)

		dataset_dict["prompt"].append(prompt)
		dataset_dict["outputs"].append(outputs)
		dataset_dict["results"].append(results)
	return Dataset.from_dict(dataset_dict)

example_ds = await generate_dataset(
	prompts=[
		"Pick an random integer from 1 to 3. Don't pick 2. Box your answer.",
		"What is 8 times 3? Box your answer.",
	],
	golds=["3", "24"],
	model=MODEL_ID,
	max_tokens=2**10,
    n=8,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**10,
    },
)
example_ds[:]

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
Creating completions: 100%|██████████| 2/2 [00:11<00:00,  5.53s/it]


{'prompt': ["Pick an random integer from 1 to 3. Don't pick 2. Box your answer.",
  'What is 8 times 3? Box your answer.'],
 'outputs': [['<think>The user says: "Pick an random integer from 1 to 3. Don\'t pick 2. Box your answer." So we need to choose either 1 or 3. It\'s a random integer from 1 to 3. But also we must not pick 2. So we can pick either 1 or 3. Let\'s pick 1 or 3. But the instructions require us to "box your answer." That likely means we need to put the number inside a box. In plain text, we can use something like [ 1 ]. Or maybe we can use ☐ style? The format: \\boxed{1} is used in LaTeX. But plain text can also do a square: ❏? The user says "Box your answer," which suggests we should delimit it visually as a box. Let\'s use \\boxed{1}. But do we need to ensure we don\'t accidentally pick 2? We\'ll pick 3, for variety. Let\'s pick 3. Then box: \\boxed{3}. But if it\'s plain text, that might not be considered a box? Typically we\'d also ensure it doesn\'t pick 2. "random

In [7]:
input_ds = ds.shuffle().select(range(2**7))
output_ds = await generate_dataset(
	prompts=input_ds["problem"],
	golds=input_ds["answer"],
	model=MODEL_ID,
	max_tokens=2**15,
    n=64,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**15,
    },
)
output_ds

INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://integrate.api.nvidia.com/v1/

Dataset({
    features: ['prompt', 'outputs', 'results'],
    num_rows: 128
})

In [8]:
output_ds.push_to_hub("EthanKim8683/reg_grpo", "128")

INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/EthanKim8683/reg_grpo "HTTP/1.1 200 OK"
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

INFO:httpx:HTTP Request: POST https://huggingface.co/api/datasets/EthanKim8683/reg_grpo/preupload/main "HTTP/1.1 200 OK"


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/EthanKim8683/reg_grpo "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/EthanKim8683/reg_grpo/revision/6e4590d28516306512a2ec373675865dc8c9526b "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/EthanKim8683/reg_grpo/tree/6e4590d28516306512a2ec373675865dc8c9526b/128?recursive=true&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/EthanKim8683/reg_grpo/tree/6e4590d28516306512a2ec373675865dc8c9526b?recursive=false&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/EthanKim8683/reg_grpo/tree/6e4590d28516306512a2ec373675865dc8c9526b/data?recursive=true&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: POST https://huggingface.co/api/validate-yaml "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://huggingface.co/api/datasets/EthanKim8683/reg_grpo/pr

CommitInfo(commit_url='https://huggingface.co/datasets/EthanKim8683/reg_grpo/commit/31412a5213ac3c14e3f0909b64b9d2e4c398a27f', commit_message='Upload dataset', commit_description='', oid='31412a5213ac3c14e3f0909b64b9d2e4c398a27f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/EthanKim8683/reg_grpo', endpoint='https://huggingface.co', repo_type='dataset', repo_id='EthanKim8683/reg_grpo'), pr_revision=None, pr_num=None)